# Person 1 – Data and Problem Framing
## Amazon Video Games Recommendation System

**Author:** Person 1 – Data and Problem Framing Lead  
**Dataset:** Amazon Video Games Reviews  
**Last updated:** March 2025

---

### Contents
1. [Business Problem Statement](#1-business-problem-statement)
2. [Dataset Selection and Justification](#2-dataset-selection-and-justification)
3. [Setup and Data Loading](#3-setup-and-data-loading)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis)
5. [Data Cleaning and Preprocessing](#5-data-cleaning-and-preprocessing)
6. [Train / Validation / Test Split](#6-train--validation--test-split)
7. [Saving Cleaned Data for Team](#7-saving-cleaned-data-for-team)

---
## 1. Business Problem Statement
> ✅ **TASK: Write the project problem statement**

### The Current Situation
> ✅ **TASK: Explain that the company currently uses random product display**

The company currently displays products to users in a **random order**. This means every user sees products regardless of their personal preferences, purchase history, or any contextual signal. There is no intelligence behind the display strategy — any user could see any product at any time with equal probability.

### Why Random Recommendations Are Weak
> ✅ **TASK: Explain why random recommendations are weak**

Random product display is effectively a non-strategy. It treats all users identically and completely ignores the rich behavioral and preference data the company has available. The problems with random recommendations include:

- **No personalisation**: Users see products irrelevant to their interests, leading to low engagement.
- **Wasted impressions**: Products are shown to users who are very unlikely to engage with them.
- **Poor customer experience**: Customers must search harder to find relevant products, reducing satisfaction and trust.
- **Missed revenue**: The company fails to leverage upsell and cross-sell opportunities that a smart recommender could surface.
- **No cold-start handling**: Even for new users, better heuristics (e.g. popularity, demographic trends) exist.

### The Project Goal
> ✅ **TASK: Explain that the goal is to recommend more relevant products to users**

Our goal is to replace the random baseline with an intelligent recommendation system that:

1. **Learns from past user behaviour** — collaborative filtering uses the purchase/review history of all users to find patterns.
2. **Uses item content** — content-based methods use product metadata (title, description, category, brand) to find similar items.
3. **Combines signals** — a hybrid approach gives the best of both worlds.
4. **Is measurably better** — we will evaluate all models against the random baseline using proper recommendation metrics (Precision@K, NDCG@K, etc.).
5. **Delivers business value** — we will quantify the expected improvement in click-through rate and conversion.

### The Business Case in One Sentence

> *If we show users the products they are most likely to buy, rather than random ones, we will increase engagement, increase sales, and improve customer satisfaction — all measurably.*

---
## 2. Dataset Selection and Justification
> ✅ **TASK: Choose the dataset**  
> ✅ **TASK: Compare possible datasets**  
> ✅ **TASK: Pick a dataset that supports collaborative filtering, content-based filtering, and hybrid methods**  
> ✅ **TASK: Make sure the dataset has user IDs, item IDs, interactions, and item metadata**

### Candidate Datasets Considered

| Dataset | Users | Items | Interactions | Has text? | Has metadata? | Verdict |
|---|---|---|---|---|---|---|
| MovieLens 1M | 6,040 | 3,706 | 1,000,209 | No | Genre only | Too small, limited metadata |
| Yelp Reviews | ~2M | ~150K | ~7M | Yes | Yes (categories, location) | Strong but non-product domain |
| Amazon Books | ~3M | ~400K | ~8M | Yes | Yes | Very large, text heavy |
| **Amazon Video Games** | **1.54M** | **72K** | **2.57M** | **Yes** | **Yes (full)** | **✅ Chosen** |

### Why We Chose the Amazon Video Games Dataset
> ✅ **TASK: Justify the dataset choice**  
> ✅ **TASK: Explain why the dataset is suitable for recommendation systems**  
> ✅ **TASK: Explain what kinds of recommendation methods it supports**  
> ✅ **TASK: Explain why it is realistic enough for business use**

**Source:** [Kaggle – Amazon Video Games Reviews](https://www.kaggle.com/datasets/gabrielfreddi/amazon-reviews-de-vdeo-games)  
**Files:** `Video_Games.json` (reviews) + `meta_Video_Games.json` (product metadata)

We chose this dataset because it satisfies every requirement of this project:

**1. Supports all recommendation approaches:**
- Explicit ratings (1–5 stars) → collaborative filtering, matrix factorisation
- Review text → content-based (TF-IDF, BERT, NER)
- Timestamps → context-aware, temporal splits
- Product metadata (title, description, brand, category) → content-based features
- Co-purchase / co-view links in metadata → graph-based signals

**2. Realistic scale:**
- 2.57 million reviews across 20 years (1997–2018)
- 1.54 million unique users, 72K unique items
- Large enough for statistically meaningful evaluation

**3. Strong business narrative:**
- Video games is an intuitive, relatable domain for a CEO presentation
- Product recommendations are a natural and well-understood business use case
- Amazon-style recommendations are the gold standard for e-commerce

**4. Data quality:**
- ~100% review text coverage
- 100% product titles, 88% descriptions, 97% categories, 95% brand info
- Verified purchase flag available for filtering

### Known Limitations

| Limitation | Impact | Mitigation |
|---|---|---|
| High sparsity (median 1 review/user) | CF harder for inactive users | Filter to users with ≥5 reviews |
| No explicit demographics | Cannot do traditional demographic filtering | Use platform (Xbox/PS/PC) from `style` field as proxy |
| Rating skew (62% are 5-star) | Evaluation metrics may be optimistic | Use ranking metrics (NDCG, MAP) in addition to RMSE |

---
## 3. Setup and Data Loading

In [ ]:
import os
import json
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Paths ──────────────────────────────────────────────────────────────────
ZIP_PATH   = '../data/raw/Amazon Video Games Reviews.zip'
RAW_DIR    = '../data/raw/'
PROC_DIR   = '../data/processed/'

os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

REVIEWS_FILE = 'Video_Games.json'
META_FILE    = 'meta_Video_Games.json'

print('Libraries loaded successfully.')
print(f'ZIP path: {ZIP_PATH}')

In [ ]:
def load_json_from_zip(zip_path, filename, max_rows=None):
    """Stream-load a newline-delimited JSON file from inside a zip archive."""
    records = []
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(filename) as f:
            for i, line in enumerate(f):
                if max_rows and i >= max_rows:
                    break
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return pd.DataFrame(records)

print('Loading reviews (this may take ~30 seconds for the full dataset)...')
reviews_raw = load_json_from_zip(ZIP_PATH, REVIEWS_FILE)
print(f'Reviews loaded:  {len(reviews_raw):,} rows')

print('Loading product metadata...')
meta_raw = load_json_from_zip(ZIP_PATH, META_FILE)
print(f'Metadata loaded: {len(meta_raw):,} rows')

In [ ]:
print('=== REVIEWS SNAPSHOT ===')
print(f'Shape: {reviews_raw.shape}')
print(f'Columns: {list(reviews_raw.columns)}')
reviews_raw.head(3)

In [ ]:
print('=== METADATA SNAPSHOT ===')
print(f'Shape: {meta_raw.shape}')
print(f'Columns: {list(meta_raw.columns)}')
meta_raw.head(3)

---
## 4. Exploratory Data Analysis
> ✅ **TASK: Perform exploratory data analysis**

### 4.1 High-Level Statistics
> ✅ **TASK: Count the number of users**  
> ✅ **TASK: Count the number of items**  
> ✅ **TASK: Count the number of interactions**

In [ ]:
n_reviews      = len(reviews_raw)
n_users        = reviews_raw['reviewerID'].nunique()
n_items        = reviews_raw['asin'].nunique()
n_meta_items   = meta_raw['asin'].nunique()
sparsity       = 1 - (n_reviews / (n_users * n_items))

ts_min = pd.to_datetime(reviews_raw['unixReviewTime'], unit='s').min()
ts_max = pd.to_datetime(reviews_raw['unixReviewTime'], unit='s').max()

print('=' * 50)
print('DATASET OVERVIEW')
print('=' * 50)
print(f'Total reviews          : {n_reviews:>12,}')
print(f'Unique users           : {n_users:>12,}')
print(f'Unique items (reviews) : {n_items:>12,}')
print(f'Unique items (metadata): {n_meta_items:>12,}')
print(f'Sparsity               : {sparsity:>12.4%}')
print(f'Date range             : {ts_min.date()} → {ts_max.date()}')
print(f'Years of data          : {(ts_max - ts_min).days // 365}')

### 4.2 Missing Values
> ✅ **TASK: Check missing values**

In [ ]:
print('=== MISSING VALUES – REVIEWS ===')
missing_reviews = reviews_raw.isnull().sum()
missing_pct     = (missing_reviews / len(reviews_raw) * 100).round(2)
missing_df      = pd.DataFrame({'missing_count': missing_reviews, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False))

In [ ]:
print('=== MISSING VALUES – METADATA ===')

# For list/dict fields, 'missing' means empty list/empty string
def count_empty(series):
    """Count truly empty values including empty lists, empty strings, None."""
    def is_empty(x):
        if x is None: return True
        if isinstance(x, float) and np.isnan(x): return True
        if isinstance(x, (list, dict)) and len(x) == 0: return True
        if isinstance(x, str) and x.strip() == '': return True
        return False
    return series.apply(is_empty).sum()

meta_missing = {col: count_empty(meta_raw[col]) for col in meta_raw.columns}
meta_missing_df = pd.DataFrame.from_dict(meta_missing, orient='index', columns=['empty_count'])
meta_missing_df['empty_pct'] = (meta_missing_df['empty_count'] / len(meta_raw) * 100).round(2)
print(meta_missing_df.sort_values('empty_pct', ascending=False).to_string())

### 4.3 Duplicate Rows
> ✅ **TASK: Check duplicate rows**

In [ ]:
# A 'duplicate' in reviews = same user reviewing the same product more than once
dup_reviews = reviews_raw.duplicated(subset=['reviewerID', 'asin'], keep=False).sum()
dup_meta    = meta_raw.duplicated(subset=['asin'], keep=False).sum()

print(f'Duplicate (user, item) pairs in reviews : {dup_reviews:,}')
print(f'Duplicate ASINs in metadata              : {dup_meta:,}')

### 4.4 Rating Distribution
> ✅ **TASK: Check the rating or interaction distribution**  
> ✅ **TASK: Create visualisation — Histogram of ratings or interactions (Fig 1)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Count bar chart
rating_counts = reviews_raw['overall'].value_counts().sort_index()
axes[0].bar(rating_counts.index.astype(int), rating_counts.values,
            color=sns.color_palette('muted'), edgecolor='white')
axes[0].set_title('Rating Distribution (Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Star Rating')
axes[0].set_ylabel('Number of Reviews')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for bar, val in zip(axes[0].patches, rating_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'{val/1e6:.2f}M', ha='center', fontsize=9)

# -- Percentage pie chart
axes[1].pie(rating_counts.values, labels=[f'{int(r)}★' for r in rating_counts.index],
            autopct='%1.1f%%', colors=sns.color_palette('muted'),
            startangle=140, pctdistance=0.8)
axes[1].set_title('Rating Distribution (%)', fontsize=13, fontweight='bold')

plt.suptitle('Fig 1 – Amazon Video Games: Rating Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/fig1_rating_distribution.png', bbox_inches='tight')
plt.show()

print(f'\nNote: {rating_counts[5.0]/len(reviews_raw):.1%} of all reviews are 5-star — strong positive skew.')

### 4.5 User Activity Distribution
> ✅ **TASK: Check the user activity distribution**  
> ✅ **TASK: Create visualisation — Distribution of interactions per user (Fig 2)**

In [ ]:
user_activity = reviews_raw['reviewerID'].value_counts()

print('User Activity Stats:')
print(f'  Min reviews/user    : {user_activity.min()}')
print(f'  Median reviews/user : {user_activity.median()}')
print(f'  Mean reviews/user   : {user_activity.mean():.2f}')
print(f'  Max reviews/user    : {user_activity.max()}')
print()
for threshold in [1, 2, 5, 10, 20]:
    n = (user_activity >= threshold).sum()
    print(f'  Users with >= {threshold:2d} reviews: {n:>8,} ({n/len(user_activity):.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Log-scale histogram
axes[0].hist(user_activity.values, bins=50, color='steelblue', edgecolor='white', log=True)
axes[0].set_title('Reviews per User (log scale)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_ylabel('Number of Users (log)')
axes[0].axvline(user_activity.median(), color='red', linestyle='--', label=f'Median={int(user_activity.median())}')
axes[0].axvline(user_activity.mean(),   color='orange', linestyle='--', label=f'Mean={user_activity.mean():.1f}')
axes[0].legend()

# -- Cumulative distribution
thresholds = range(1, 51)
pct_users  = [(user_activity >= t).sum() / len(user_activity) * 100 for t in thresholds]
axes[1].plot(thresholds, pct_users, marker='o', markersize=3, color='steelblue')
axes[1].axvline(5, color='red', linestyle='--', label='Threshold = 5')
axes[1].set_title('% Users With At Least N Reviews', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Minimum reviews per user (N)')
axes[1].set_ylabel('% of users retained')
axes[1].legend()

plt.suptitle('Fig 2 – User Activity Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/fig2_user_activity.png', bbox_inches='tight')
plt.show()

### 4.6 Item Popularity Distribution
> ✅ **TASK: Check the item popularity distribution**  
> ✅ **TASK: Create visualisation — Distribution of interactions per item (Fig 3)**

In [ ]:
item_popularity = reviews_raw['asin'].value_counts()

print('Item Popularity Stats:')
print(f'  Min reviews/item    : {item_popularity.min()}')
print(f'  Median reviews/item : {item_popularity.median()}')
print(f'  Mean reviews/item   : {item_popularity.mean():.2f}')
print(f'  Max reviews/item    : {item_popularity.max()}')
print()
for threshold in [1, 5, 10, 20, 50]:
    n = (item_popularity >= threshold).sum()
    print(f'  Items with >= {threshold:2d} reviews: {n:>7,} ({n/len(item_popularity):.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Log-scale histogram
axes[0].hist(item_popularity.values, bins=50, color='coral', edgecolor='white', log=True)
axes[0].set_title('Reviews per Item (log scale)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_ylabel('Number of Items (log)')
axes[0].axvline(item_popularity.median(), color='red', linestyle='--', label=f'Median={int(item_popularity.median())}')
axes[0].axvline(item_popularity.mean(),   color='orange', linestyle='--', label=f'Mean={item_popularity.mean():.1f}')
axes[0].legend()

# -- Long-tail illustration
sorted_pop = item_popularity.values
axes[1].plot(range(1, len(sorted_pop)+1), sorted_pop, color='coral')
axes[1].set_title('Item Popularity Long-Tail Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Item rank (most to least popular)')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_yscale('log')
axes[1].set_xscale('log')

plt.suptitle('Fig 3 – Item Popularity Distribution', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/fig3_item_popularity.png', bbox_inches='tight')
plt.show()

### 4.7 Top 20 Most Popular Items
> ✅ **TASK: Create visualisation — Top 20 most popular items (Fig 4)**

In [ ]:
# Join with metadata to get titles
top20_asins  = item_popularity.head(20).reset_index()
top20_asins.columns = ['asin', 'review_count']

meta_titles = meta_raw[['asin', 'title']].drop_duplicates('asin')
top20        = top20_asins.merge(meta_titles, on='asin', how='left')
top20['title_short'] = top20['title'].fillna('Unknown').str[:50]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top20['title_short'][::-1], top20['review_count'][::-1],
               color=sns.color_palette('muted', 20))
ax.set_xlabel('Number of Reviews')
ax.set_title('Fig 4 – Top 20 Most Reviewed Video Game Products', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top20['review_count'][::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/processed/fig4_top20_items.png', bbox_inches='tight')
plt.show()

print(top20[['asin', 'title_short', 'review_count']].to_string(index=False))

### 4.8 Review Volume Over Time

In [ ]:
reviews_raw['review_date'] = pd.to_datetime(reviews_raw['unixReviewTime'], unit='s')
reviews_raw['year_month']  = reviews_raw['review_date'].dt.to_period('M')

monthly_counts = reviews_raw.groupby('year_month').size()
monthly_counts.index = monthly_counts.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_counts.index, monthly_counts.values, linewidth=1, color='steelblue')
ax.fill_between(monthly_counts.index, monthly_counts.values, alpha=0.2, color='steelblue')
ax.set_title('Fig 5 – Review Volume Over Time (Monthly)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Reviews')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../data/processed/fig5_reviews_over_time.png', bbox_inches='tight')
plt.show()

### 4.9 Sparsity and Interaction Matrix Summary
> ✅ **TASK: Calculate sparsity**

In [ ]:
total_possible  = n_users * n_items
actual_entries  = n_reviews
sparsity        = 1 - actual_entries / total_possible

print('─' * 50)
print('INTERACTION MATRIX SUMMARY')
print('─' * 50)
print(f'Matrix dimensions   : {n_users:,} users × {n_items:,} items')
print(f'Total possible cells: {total_possible:,.0f}')
print(f'Filled cells        : {actual_entries:,}')
print(f'Sparsity            : {sparsity:.6%}')
print()
print('Interpretation: Only {:.4f}% of all user-item pairs have a rating.'.format((1-sparsity)*100))
print('This is typical for real-world recommender datasets.')
print('It reinforces the need for careful filtering before collaborative filtering.')

### 4.10 Metadata Overview
> ✅ **TASK: Create visualisation — Metadata examples**

In [ ]:
# Sample metadata entries
print('=== SAMPLE METADATA ENTRIES ===')
sample_meta = meta_raw[['asin', 'title', 'brand', 'category', 'price', 'description']].dropna(subset=['title']).head(5)
for _, row in sample_meta.iterrows():
    print(f"\nASIN  : {row['asin']}")
    print(f"Title : {str(row['title'])[:80]}")
    print(f"Brand : {row['brand']}")
    print(f"Cat   : {row['category']}")
    print(f"Price : {row['price']}")
    desc = row['description']
    if isinstance(desc, list) and desc:
        print(f"Desc  : {str(desc[0])[:100]}")

---
## 5. Data Cleaning and Preprocessing
> ✅ **TASK: Clean the dataset**

### 5.1 Reviews – Cleaning
> ✅ **TASK: Handle missing values** (drop rows missing user/item/rating)  
> ✅ **TASK: Remove duplicates** (duplicate user-item pairs, keep most recent)  
> ✅ **TASK: Standardise column names** (renamed to user_id, item_id, rating, timestamp, etc.)

In [ ]:
print(f'Reviews before cleaning: {len(reviews_raw):,}')

reviews = reviews_raw.copy()

# 1. Drop rows missing essential fields
reviews = reviews.dropna(subset=['reviewerID', 'asin', 'overall'])
print(f'After dropping rows missing user/item/rating: {len(reviews):,}')

# 2. Remove duplicate (user, item) pairs — keep most recent
reviews = reviews.sort_values('unixReviewTime', ascending=False)
reviews = reviews.drop_duplicates(subset=['reviewerID', 'asin'], keep='first')
print(f'After removing duplicate (user, item) pairs: {len(reviews):,}')

# 3. Standardise column names
reviews = reviews.rename(columns={
    'reviewerID'   : 'user_id',
    'asin'         : 'item_id',
    'overall'      : 'rating',
    'unixReviewTime': 'timestamp',
    'reviewText'   : 'review_text',
    'summary'      : 'review_summary',
    'reviewTime'   : 'review_date_str',
    'reviewerName' : 'user_name',
})

# 4. Ensure correct dtypes
reviews['rating']    = reviews['rating'].astype(float)
reviews['timestamp'] = reviews['timestamp'].astype(int)
reviews['review_date'] = pd.to_datetime(reviews['timestamp'], unit='s')

# 5. Keep only useful columns
reviews = reviews[['user_id', 'item_id', 'rating', 'timestamp', 'review_date',
                    'review_text', 'review_summary', 'verified']]

print(f'Reviews after cleaning: {len(reviews):,}')
print(f'Columns: {list(reviews.columns)}')
reviews.head(3)

### 5.2 Filtering – Minimum Activity Threshold
> ✅ **TASK: Filter very inactive users if needed**  
> ✅ **TASK: Filter very unpopular items if needed**

We apply a **5-core filter**: keep only users with ≥5 reviews and items with ≥5 reviews.  
This is standard practice in recommendation systems research to ensure:
- Collaborative filtering has enough signal per user
- Evaluation metrics are meaningful
- Memory and compute are manageable

In [ ]:
MIN_USER_REVIEWS = 5
MIN_ITEM_REVIEWS = 5

def apply_kcore_filter(df, min_user=5, min_item=5, max_iterations=10):
    """Iteratively apply k-core filter until convergence."""
    for i in range(max_iterations):
        n_before = len(df)
        user_counts = df['user_id'].value_counts()
        item_counts = df['item_id'].value_counts()
        valid_users = user_counts[user_counts >= min_user].index
        valid_items = item_counts[item_counts >= min_item].index
        df = df[df['user_id'].isin(valid_users) & df['item_id'].isin(valid_items)]
        if len(df) == n_before:
            print(f'  Converged after {i+1} iteration(s).')
            break
    return df

print(f'Before filtering: {len(reviews):,} interactions, '
      f'{reviews["user_id"].nunique():,} users, '
      f'{reviews["item_id"].nunique():,} items')

reviews_filtered = apply_kcore_filter(reviews, MIN_USER_REVIEWS, MIN_ITEM_REVIEWS)

n_filtered     = len(reviews_filtered)
n_users_f      = reviews_filtered['user_id'].nunique()
n_items_f      = reviews_filtered['item_id'].nunique()
sparsity_f     = 1 - n_filtered / (n_users_f * n_items_f)

print(f'After  filtering: {n_filtered:,} interactions, '
      f'{n_users_f:,} users, '
      f'{n_items_f:,} items')
print(f'Retained: {n_filtered/len(reviews):.1%} of interactions, '
      f'{n_users_f/reviews["user_id"].nunique():.1%} of users, '
      f'{n_items_f/reviews["item_id"].nunique():.1%} of items')
print(f'New sparsity: {sparsity_f:.4%}')

### 5.3 Metadata – Cleaning
> ✅ **TASK: Standardise column names in metadata**  
> ✅ **TASK: Handle missing values in metadata** (price, description, features, categories)

In [ ]:
meta = meta_raw.copy()

# Keep only items present in filtered reviews
meta = meta[meta['asin'].isin(reviews_filtered['item_id'])]

# Standardise column names
meta = meta.rename(columns={'asin': 'item_id'})

# Clean price column (remove HTML/symbols)
def clean_price(p):
    if not isinstance(p, str) or p.strip() == '': return np.nan
    import re
    match = re.search(r'[\d]+\.?[\d]*', p.replace(',', ''))
    return float(match.group()) if match else np.nan

meta['price_clean'] = meta['price'].apply(clean_price)

# Flatten description (list → string)
meta['description_text'] = meta['description'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else (x if isinstance(x, str) else '')
)

# Flatten features (list → string)
meta['features_text'] = meta['feature'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else ''
)

# Top-level category
meta['main_category'] = meta['category'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'Unknown'
)

# Sub-category (2nd level)
meta['sub_category'] = meta['category'].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) > 1 else 'Unknown'
)

# Combined text field for content-based models (Person 3 will use this)
meta['content_text'] = (
    meta['title'].fillna('') + ' ' +
    meta['description_text'].fillna('') + ' ' +
    meta['features_text'].fillna('') + ' ' +
    meta['brand'].fillna('') + ' ' +
    meta['main_category'].fillna('')
)

# Keep relevant columns
meta = meta[['item_id', 'title', 'brand', 'main_category', 'sub_category',
             'price_clean', 'description_text', 'features_text',
             'content_text', 'also_buy', 'also_view']]

print(f'Cleaned metadata: {len(meta):,} items')
print(f'Columns: {list(meta.columns)}')
meta.head(3)

### 5.4 Build Clean Output Tables
> ✅ **TASK: Create cleaned files/tables — User table**  
> ✅ **TASK: Create cleaned files/tables — Item table**  
> ✅ **TASK: Create cleaned files/tables — Interaction table**  
> ✅ **TASK: Create cleaned files/tables — Metadata table / feature-ready item table**

In [ ]:
# ── User Table ─────────────────────────────────────────────────────────────
user_table = reviews_filtered.groupby('user_id').agg(
    n_reviews       = ('rating', 'count'),
    avg_rating      = ('rating', 'mean'),
    first_review    = ('timestamp', 'min'),
    last_review     = ('timestamp', 'max'),
).reset_index()
user_table['avg_rating'] = user_table['avg_rating'].round(2)

print(f'User table: {len(user_table):,} users')
user_table.head(3)

In [ ]:
# ── Item Table ─────────────────────────────────────────────────────────────
item_stats = reviews_filtered.groupby('item_id').agg(
    n_reviews       = ('rating', 'count'),
    avg_rating      = ('rating', 'mean'),
).reset_index()
item_stats['avg_rating'] = item_stats['avg_rating'].round(2)

item_table = item_stats.merge(meta[['item_id', 'title', 'brand', 'main_category',
                                     'sub_category', 'price_clean']], on='item_id', how='left')

print(f'Item table: {len(item_table):,} items')
item_table.head(3)

In [ ]:
# ── Interaction Table ──────────────────────────────────────────────────────
interaction_table = reviews_filtered[['user_id', 'item_id', 'rating', 'timestamp',
                                       'review_date', 'verified']].copy()

print(f'Interaction table: {len(interaction_table):,} rows')
interaction_table.head(3)

---
## 6. Train / Validation / Test Split
> ✅ **TASK: Split the data**  
> ✅ **TASK: Create train/validation/test split**  
> ✅ **TASK: Make sure there is no data leakage**  
> ✅ **TASK: Use a time-based split (timestamps exist)**  
> ✅ **TASK: Explain why this split method was chosen**

### Methodology: Time-Based Split

Because our dataset has timestamps spanning 20 years, we use a **time-based split** rather than a random split. This is critical because:

- A random split would allow future reviews to "leak" into training data — the model would be trained on information it could not have known at prediction time.
- A time-based split simulates real deployment: we train on past behaviour and evaluate on future behaviour.
- This is considered best practice in the recommendation systems literature.

**Split strategy:**
- **Training set**: First 70% of time (by timestamp per user)
- **Validation set**: Next 15% (for hyperparameter tuning — Person 2 and 3 will use this)
- **Test set**: Last 15% (held out for final evaluation — Person 4 will use this)

We split **per user** to ensure every user appears in training.

In [ ]:
# Sort by timestamp
interactions_sorted = interaction_table.sort_values('timestamp').reset_index(drop=True)

def time_based_split(df, train_frac=0.70, val_frac=0.15):
    """
    Per-user time-based split.
    For each user, sort their interactions by time and assign:
      - first train_frac → train
      - next  val_frac   → validation
      - rest             → test
    Users with too few interactions are kept entirely in train.
    """
    train_idx, val_idx, test_idx = [], [], []

    for _, user_df in df.groupby('user_id'):
        user_df = user_df.sort_values('timestamp')
        n = len(user_df)

        if n < 3:
            train_idx.extend(user_df.index.tolist())
            continue

        n_train = max(1, int(n * train_frac))
        n_val   = max(1, int(n * val_frac))

        train_idx.extend(user_df.iloc[:n_train].index.tolist())
        val_idx.extend(user_df.iloc[n_train:n_train + n_val].index.tolist())
        test_idx.extend(user_df.iloc[n_train + n_val:].index.tolist())

    train = df.loc[train_idx].copy()
    val   = df.loc[val_idx].copy()
    test  = df.loc[test_idx].copy()
    return train, val, test

print('Creating train/validation/test split...')
train, val, test = time_based_split(interactions_sorted)

print(f'\nSplit results:')
print(f'  Train : {len(train):>8,} interactions ({len(train)/len(interactions_sorted):.1%})')
print(f'  Val   : {len(val):>8,} interactions ({len(val)/len(interactions_sorted):.1%})')
print(f'  Test  : {len(test):>8,} interactions ({len(test)/len(interactions_sorted):.1%})')
print()
print(f'  Train date range: {pd.to_datetime(train["timestamp"].min(), unit="s").date()} → {pd.to_datetime(train["timestamp"].max(), unit="s").date()}')
print(f'  Val   date range: {pd.to_datetime(val["timestamp"].min(), unit="s").date()} → {pd.to_datetime(val["timestamp"].max(), unit="s").date()}')
print(f'  Test  date range: {pd.to_datetime(test["timestamp"].min(), unit="s").date()} → {pd.to_datetime(test["timestamp"].max(), unit="s").date()}')

In [ ]:
# Verify no data leakage: no user appears in val/test but NOT in train
train_users = set(train['user_id'])
val_users   = set(val['user_id'])
test_users  = set(test['user_id'])

val_only  = val_users - train_users
test_only = test_users - train_users

print('Data leakage check:')
print(f'  Users in val  but NOT in train: {len(val_only):,}  (should be 0 or small)')
print(f'  Users in test but NOT in train: {len(test_only):,}  (should be 0 or small)')
print()

# Visualise split over time
fig, ax = plt.subplots(figsize=(14, 4))
for split, color, label in [(train, 'steelblue', 'Train'),
                             (val,   'orange',    'Validation'),
                             (test,  'red',       'Test')]:
    monthly = split.copy()
    monthly['ym'] = pd.to_datetime(monthly['timestamp'], unit='s').dt.to_period('M').dt.to_timestamp()
    cnt = monthly.groupby('ym').size()
    ax.fill_between(cnt.index, cnt.values, alpha=0.5, label=label, color=color)

ax.set_title('Fig 6 – Train / Validation / Test Split Over Time', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Interactions')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/fig6_train_val_test_split.png', bbox_inches='tight')
plt.show()

---
## 7. Saving Cleaned Data for Team
> ✅ **TASK: Create cleaned files/tables — all 7 output files saved for the team**

All outputs are saved to `data/processed/` for use by other team members.

In [ ]:
# Save all cleaned tables
print('Saving cleaned data...')

interaction_table.to_csv('../data/processed/interactions_clean.csv', index=False)
user_table.to_csv('../data/processed/users.csv', index=False)
item_table.to_csv('../data/processed/items.csv', index=False)
meta.to_csv('../data/processed/metadata_clean.csv', index=False)

train.to_csv('../data/processed/train.csv', index=False)
val.to_csv('../data/processed/val.csv', index=False)
test.to_csv('../data/processed/test.csv', index=False)

print('✓ interactions_clean.csv')
print('✓ users.csv')
print('✓ items.csv')
print('✓ metadata_clean.csv')
print('✓ train.csv')
print('✓ val.csv')
print('✓ test.csv')
print()
print('All data saved to data/processed/')

In [ ]:
# Final summary for team
print('=' * 60)
print('PERSON 1 – FINAL SUMMARY FOR TEAM')
print('=' * 60)
print()
print('DATASET')
print(f'  Source       : Amazon Video Games Reviews (Kaggle)')
print(f'  Raw reviews  : 2,565,349')
print(f'  Raw items    : 71,982')
print()
print('AFTER CLEANING + 5-CORE FILTER')
print(f'  Interactions : {len(interaction_table):,}')
print(f'  Users        : {len(user_table):,}')
print(f'  Items        : {len(item_table):,}')
print(f'  Sparsity     : {1 - len(interaction_table)/(len(user_table)*len(item_table)):.4%}')
print()
print('TRAIN / VAL / TEST SPLIT (time-based, per user)')
print(f'  Train : {len(train):,}')
print(f'  Val   : {len(val):,}')
print(f'  Test  : {len(test):,}')
print()
print('OUTPUT FILES → data/processed/')
print('  interactions_clean.csv  – full cleaned interactions')
print('  users.csv               – user activity stats')
print('  items.csv               – item stats + metadata')
print('  metadata_clean.csv      – full cleaned metadata')
print('  train.csv               – training set')
print('  val.csv                 – validation set')
print('  test.csv                – test set (do not touch until final eval!)')